In [1]:
import pandas as pd
import re
from collections import Counter

# Step 1: Load the CSV file
def load_and_clean_csv(file_path):
    # Load the data
    data = pd.read_csv(file_path)

    # Ensure column names and data are clean
    data = data.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    # Drop rows with missing essential data
    data.dropna(subset=['request', 'label'], inplace=True)

    return data

# Step 2: Detection rules for various attack types
def detect_attacks(data, brute_force_threshold=5, dos_rate_threshold=100):
    # SQL Injection Detection
    sql_injection_pattern = r".*(UNION|SELECT|INSERT|DELETE|UPDATE).*"
    data['potential_sql_injection'] = data['request'].str.contains(sql_injection_pattern, case=False, na=False)

    # XSS Detection
    xss_pattern = r".*(<script>|%3Cscript%3E).*"
    data['potential_xss'] = data['request'].str.contains(xss_pattern, case=False, na=False)

    # Directory Traversal Detection
    dir_traversal_pattern = r".*(\..\/|\.\.\\).*"
    data['potential_dir_traversal'] = data['request'].str.contains(dir_traversal_pattern, case=False, na=False)

    # Brute Force Detection (using client_ip and timestamp for simplicity)
    ip_attempts = Counter(data['client_ip'])
    data['potential_brute_force'] = data['client_ip'].map(lambda ip: ip_attempts[ip] > brute_force_threshold)

    # DoS Detection (based on the rate of requests per IP within a short time frame)
    data['request_rate'] = data.groupby('client_ip')['client_ip'].transform('count')
    data['potential_dos'] = data['request_rate'] > dos_rate_threshold

    return data

# Step 3: Relabel the data based on detected patterns
def relabel_data(data):
    def determine_label(row):
        if row['potential_sql_injection']:
            return 'SQL Injection'
        if row['potential_xss']:
            return 'XSS'
        if row['potential_dir_traversal']:
            return 'Directory Traversal'
        if row['potential_brute_force']:
            return 'Brute Force'
        if row['potential_dos']:
            return 'DoS'
        return row['label']

    data['corrected_label'] = data.apply(determine_label, axis=1)
    return data

# Step 4: Save the corrected data
def save_corrected_data(data, output_path):
    # Drop helper columns if they exist
    data = data.drop(columns=['potential_sql_injection', 'potential_xss', 'potential_dir_traversal', 'potential_brute_force', 'potential_dos', 'request_rate'], errors='ignore')
    
    # Save the data to a new CSV file
    data.to_csv(output_path, index=False)

# Main function to process and relabel the dataset
def main():
    input_file = './dataset/labeled_logs.csv'  # Replace with your input file path
    output_file = './dataset/corrected_labeled_logs.csv'  # Replace with your desired output file path

    # Load and clean the dataset
    data = load_and_clean_csv(input_file)

    # Detect various attacks
    data = detect_attacks(data)

    # Relabel the dataset
    data = relabel_data(data)

    # Save the corrected dataset
    save_corrected_data(data, output_file)
    print(f"Corrected dataset saved to {output_file}")

# For running in Jupyter Notebook
if __name__ == "__main__":
    main()


C:\Users\TAMA\AppData\Local\Temp\ipykernel_9296\1100426399.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  data = data.applymap(lambda x: x.strip() if isinstance(x, str) else x)
C:\Users\TAMA\AppData\Local\Temp\ipykernel_9296\1100426399.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  data['potential_sql_injection'] = data['request'].str.contains(sql_injection_pattern, case=False, na=False)
C:\Users\TAMA\AppData\Local\Temp\ipykernel_9296\1100426399.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  data['potential_xss'] = data['request'].str.contains(xss_pattern, case=False, na=False)
C:\Users\TAMA\AppData\Local\Temp\ipykernel_9296\1100426399.py:30: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups,

Corrected dataset saved to ./dataset/corrected_labeled_logs.csv
